# gistemp5 vs gistemp4.0 — Step 2 Comparison

Validates that gistemp5's step 2 output is byte-for-byte identical to gistemp4.0,
and shows the global temperature signal produced by both pipelines.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Works whether the notebook is run from scripts/ or from the repo root
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if not os.path.exists(os.path.join(REPO_ROOT, 'parameters')):
    REPO_ROOT = os.getcwd()
sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)  # logger and steps write relative paths (logs/, input/)

from parameters.constants import START_YEAR, END_YEAR

print(f"Repo root : {REPO_ROOT}")
print(f"Year range: {START_YEAR}–{END_YEAR}")

## Load step 2 outputs

gistemp4.0's output is read from the existing cache.
gistemp5 is run once and cached alongside it.

In [ ]:
from tools import cache as step_cache

V4_CACHE = os.path.join(REPO_ROOT, 'gistemp4.0', 'tmp', 'step2_cache.parquet')

print("Loading gistemp4.0 step 2 output...")
df4 = pd.read_parquet(V4_CACHE)

print("Loading gistemp5 step 2 output...")
df5 = step_cache.load('step2', START_YEAR, END_YEAR)
if df5 is None:
    print("  Cache miss — running step0 → step1 → step2 (~10 min, cached afterwards)...")
    from parameters.data import GHCN_TEMP_URL, GHCN_META_URL, STRANGE_URL, BRIGHTNESS_URL
    from steps.step0 import step0
    from steps.step1 import step1
    from steps.step2 import step2

    df0 = step_cache.load('step0', START_YEAR, END_YEAR)
    if df0 is None:
        df0 = step0(GHCN_TEMP_URL, GHCN_META_URL, START_YEAR, END_YEAR)
        step_cache.save(df0, 'step0', START_YEAR, END_YEAR)

    df1 = step_cache.load('step1', START_YEAR, END_YEAR)
    if df1 is None:
        df1 = step1(df0, STRANGE_URL, START_YEAR, END_YEAR)
        step_cache.save(df1, 'step1', START_YEAR, END_YEAR)

    df5 = step2(df1, GHCN_META_URL, BRIGHTNESS_URL, START_YEAR, END_YEAR)
    step_cache.save(df5, 'step2', START_YEAR, END_YEAR)
else:
    print("  Loaded from cache.")

# Shared stations and monthly columns
meta_cols = {'Latitude', 'Longitude', '__LastGHCNYear__'}
tc4 = [c for c in df4.columns if c not in meta_cols]
tc5 = [c for c in df5.columns if c not in meta_cols]
shared_cols = sorted(set(tc4) & set(tc5), key=lambda c: (int(c.split('_')[1]), int(c.split('_')[0])))
shared_sids = sorted(set(df4.index) & set(df5.index))

print(f"\ngistemp4.0 : {len(df4):,} stations")
print(f"gistemp5   : {len(df5):,} stations")
print(f"Shared     : {len(shared_sids):,} stations, {len(shared_cols):,} monthly columns")

## Identity validation

In [ ]:
a = df5.loc[shared_sids, shared_cols].astype(float)
b = df4.loc[shared_sids, shared_cols].astype(float)
diff = (a - b).abs()
both_valid = ~a.isna() & ~b.isna()

nan_mismatch    = int((a.isna() != b.isna()).sum().sum())
cells_compared  = int(both_valid.sum().sum())
cells_differ    = int((diff[both_valid] > 1e-4).sum().sum())
max_diff        = float(diff.max().max())

summary = pd.DataFrame({
    'Metric': [
        'Shared stations',
        'Monthly cells compared',
        'NaN mismatches',
        'Cells differing > 1e-4 °C',
        'Max absolute difference (°C)',
    ],
    'Value': [
        f"{len(shared_sids):,}",
        f"{cells_compared:,}",
        f"{nan_mismatch:,}",
        f"{cells_differ:,}",
        f"{max_diff:.2e}",
    ],
}).set_index('Metric')

display(summary)

if cells_differ == 0 and nan_mismatch == 0:
    print("\n✓ Outputs are IDENTICAL.")
else:
    print(f"\n✗ {cells_differ:,} differing cells — investigate further.")

## Global temperature signal

Unweighted mean of all valid station readings per year.
Both pipelines produce the same signal; the difference panel should be zero everywhere.

In [ ]:
def annual_global_mean(df, start_year, end_year):
    years = range(start_year, end_year + 1)
    means, counts = [], []
    for yr in years:
        cols = [f'{m}_{yr}' for m in range(1, 13) if f'{m}_{yr}' in df.columns]
        if cols:
            vals = df[cols].values
            means.append(np.nanmean(vals))
            counts.append(int(np.any(~np.isnan(vals), axis=1).sum()))
        else:
            means.append(np.nan)
            counts.append(0)
    idx = list(years)
    return pd.Series(means, index=idx), pd.Series(counts, index=idx)

mean4, cnt4 = annual_global_mean(df4, START_YEAR, END_YEAR)
mean5, cnt5 = annual_global_mean(df5, START_YEAR, END_YEAR)

fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
fig.suptitle(f'Step 2 Output: gistemp5 vs gistemp4.0  ({START_YEAR}–{END_YEAR})',
             fontsize=14, fontweight='bold')

BLUE  = '#1565C0'
RED   = '#E53935'
GREEN = '#2E7D32'

# ── Global mean temperature ──────────────────────────────────────
ax = axes[0]
ax.plot(mean4.index, mean4.values, color=BLUE, lw=2.5, label='gistemp4.0', zorder=3)
ax.plot(mean5.index, mean5.values, color=RED,  lw=1.5, ls='--',
        label='gistemp5 (identical)', zorder=4)
ax.set_ylabel('Mean Temperature (°C)', fontsize=11)
ax.set_title('Global Mean Station Temperature (unweighted)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.25)

# ── Absolute difference ──────────────────────────────────────────
ax = axes[1]
abs_diff = (mean5 - mean4).abs()
ax.plot(abs_diff.index, abs_diff.values, color=GREEN, lw=1.5)
ax.axhline(0, color='black', lw=0.8, ls='--', alpha=0.5)
ax.set_ylabel('|Δ| (°C)', fontsize=11)
ax.set_title('Absolute Difference: |gistemp5 − gistemp4.0|', fontsize=12)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.2e'))
ax.grid(alpha=0.25)

# ── Station count ────────────────────────────────────────────────
ax = axes[2]
ax.fill_between(cnt4.index, cnt4.values, color=BLUE, alpha=0.4, label='gistemp4.0')
ax.fill_between(cnt5.index, cnt5.values, color=RED,  alpha=0.35, label='gistemp5')
ax.set_ylabel('Active Stations', fontsize=11)
ax.set_xlabel('Year', fontsize=11)
ax.set_title('Active Station Count per Year', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.25)
ax.set_xlim(START_YEAR, END_YEAR)

plt.tight_layout()
out = os.path.join(REPO_ROOT, 'scripts', 'step2_comparison.png')
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f"Figure saved → {out}")